# Final Visual Set and Figure Mapping

This notebook produces many exploratory plots. For the **final project presentation and report**, we focus on the following 5 core visualizations, each directly answering a proposal question:

- **Figure 1 – Category Bar Chart (Section 4)**: Top 15 electronics categories by product count, colored by average rating. This supports the question *"Which electronics categories generate the highest sales volume and where do highly rated products concentrate?"*
- **Figure 2 – Discount vs Sales Scatter (Section 5, VIZ 3)**: Discount percentage vs log monthly purchases with a LOWESS trendline and Best Seller coloring, answering *"How does discount percentage relate to product sales performance?"*
- **Figure 3 – Discount Bucket Bar Comparison (Section 5, VIZ 4)**: Side‑by‑side bars showing average monthly purchases and average rating across discount ranges, highlighting the *"moderate discounts vs extreme discounts"* trade‑off.
- **Figure 4 – Best Seller Price Distribution (Section 5, VIZ 5)**: Overlapping price histograms for Best Seller vs non‑Best Seller products with median lines, addressing *"How do prices differ between Best Seller and non‑Best Seller products?"*
- **Figure 5 – Correlation Heatmap (Section 7, VIZ 8)**: Annotated correlation matrix of key variables (price, discount, ratings, reviews, purchases, badges), used to summarize *"What relationships exist between ratings, reviews, pricing, discounts, and sales?"*

All other charts in this notebook remain as **supporting visuals** for additional context, but only Figures 1–5 above should be treated as primary figures in the written report and final presentation.

# Do Discounts Really Drive Sales?
## A Visual Analysis of Amazon Electronics Products

This notebook explores how pricing strategies, discounts, and product attributes relate to sales performance on Amazon electronics products.

**Dataset:** [Amazon Electronics Products Sales Dataset – 2025](https://www.kaggle.com/datasets/ikramshah512/amazon-products-sales-dataset-42k-items-2025/data)  
**Team:** Om Bagadia (219273994) | Prabhleen Luthra (219617109) | Lavanya Kumar (219148790)

| Variable | Description |
|---|---|
| `product_title` | Full product name |
| `product_rating` | Average customer rating out of 5 |
| `total_reviews` | Total number of customer reviews |
| `purchased_last_month` | Units purchased last month (primary sales metric) |
| `discounted_price` | Selling price after discount (USD) |
| `original_price` | Original listed price (USD) |
| `discount_percentage` | Percentage discount applied |
| `is_best_seller` | Best Seller badge status |
| `is_sponsored` | Sponsored or Organic listing |
| `has_coupon` | Coupon availability |
| `product_category` | Product category |

## Importing the Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('default')

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'browser'

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('All libraries imported successfully!')

All libraries imported successfully!


## 1. Dataset Description

**Source:** Amazon Electronics Products Sales Dataset (2025) — Kaggle

**Why this dataset is suitable:**
- 42,000+ real Amazon product listings — large enough for meaningful patterns
- Combines pricing, discount, and engagement metrics — enables multi-dimensional analysis
- Includes categorical flags (Best Seller, Sponsored, Coupon) — supports group comparisons
- `purchased_last_month` provides a direct sales signal — stronger than using reviews alone
- Mix of numeric and categorical variables — supports diverse chart types

In [2]:
# ── LOAD DATASET ──────────────────────────────────────────
df_raw = pd.read_csv('amazon_products_sales_data_cleaned.csv')

print(f'Shape:   {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')
display(df_raw.head(3))

Shape:   (42675, 17)
Columns: ['product_title', 'product_rating', 'total_reviews', 'purchased_last_month', 'discounted_price', 'original_price', 'is_best_seller', 'is_sponsored', 'has_coupon', 'buy_box_availability', 'delivery_date', 'sustainability_tags', 'product_image_url', 'product_page_url', 'data_collected_at', 'product_category', 'discount_percentage']


,product_title,product_rating,total_reviews,purchased_last_month,discounted_price,original_price,is_best_seller,is_sponsored,has_coupon,buy_box_availability,delivery_date,sustainability_tags,product_image_url,product_page_url,data_collected_at,product_category,discount_percentage
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.60,375.00,300.00,89.68,159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,2025-09-01,Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,2025-08-21 11:14:29,Phones,43.60
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.30,2457.00,6000.00,9.99,15.99,No Badge,Sponsored,No Coupon,Add to cart,2025-08-29,NaN,https://m.media-amazon.com/images/I/61nbF6aVIP...,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,2025-08-21 11:14:29,Laptops,37.52
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.60,3044.00,2000.00,314.00,349.00,No Badge,Sponsored,No Coupon,Add to cart,2025-09-01,NaN,https://m.media-amazon.com/images/I/61h78MEXoj...,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,2025-08-21 11:14:29,Laptops,10.03


## 2. Data Preparation & Assumptions

**Cleaning steps:**
1. Converted categorical flags (Best Seller, Sponsored, Coupon) to binary 0/1
2. Removed rows with missing prices, ratings, or category
3. Removed invalid prices (zero or negative)
4. Removed extreme price outliers (outside 1st–99th percentile)
5. Filled missing discount % with 0
6. Created discount buckets for grouped analysis
7. Applied log transforms to skewed columns

**Assumptions:**
- `purchased_last_month` = primary sales proxy (Amazon rounds to nearest 100/1000)
- `total_reviews` = secondary sales signal
- discount_percentage = 0 means full price, no discount applied

In [3]:
# ── CLEANING ──────────────────────────────────────────────
df = df_raw.copy()
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_', regex=False)

# Binary flags
df['is_best_seller_bin'] = (df['is_best_seller'].astype(str).str.strip() == 'Best Seller').astype(int)
df['is_sponsored_bin']   = (df['is_sponsored'].astype(str).str.strip() == 'Sponsored').astype(int)
df['has_coupon_bin']     = (~df['has_coupon'].astype(str).str.strip().str.contains('No Coupon', na=True)).astype(int)
df['buy_box_bin']        = (df['buy_box_availability'].astype(str).str.strip() == 'Add to cart').astype(int)

# Numeric types
for col in ['product_rating','total_reviews','purchased_last_month','discounted_price','original_price','discount_percentage']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop missing core columns
rows_before = len(df)
df_clean = df.dropna(subset=['product_rating','discounted_price','original_price','product_category']).copy()
df_clean = df_clean[df_clean['original_price'] > 0].copy()

# Price outliers
p1  = df_clean['original_price'].quantile(0.01)
p99 = df_clean['original_price'].quantile(0.99)
df_clean = df_clean[(df_clean['original_price'] >= p1) & (df_clean['original_price'] <= p99)].copy()

# Fill missing discount
df_clean['discount_percentage'] = df_clean['discount_percentage'].fillna(0)

# Discount buckets
df_clean['discount_bucket'] = pd.cut(
    df_clean['discount_percentage'],
    bins=[-1, 0, 10, 20, 30, 40, 50, 100],
    labels=['No Discount','1-10%','11-20%','21-30%','31-40%','41-50%','50%+']
)

# Log transforms
df_clean['log_reviews'] = np.log10(df_clean['total_reviews'].replace(0, np.nan))
df_clean['log_price']   = np.log10(df_clean['original_price'].replace(0, np.nan))
df_clean['log_sales']   = np.log10(df_clean['purchased_last_month'].replace(0, np.nan))

# Top categories
top10_cats = df_clean['product_category'].value_counts().head(10).index.tolist()
df_top     = df_clean[df_clean['product_category'].isin(top10_cats)].copy()

print(f'Rows before: {rows_before:,}')
print(f'Rows after:  {len(df_clean):,}')
print(f'Removed:     {rows_before - len(df_clean):,}')
print()
print('Best Seller:', df_clean['is_best_seller'].value_counts().to_dict())
print('Sponsored:  ', df_clean['is_sponsored'].value_counts().to_dict())
display(df_clean[['product_rating','total_reviews','purchased_last_month','original_price','discount_percentage']].describe())

Rows before: 42,675
Rows after:  38,833
Removed:     3,842

Best Seller: {'No Badge': 37058, "Amazon's": 736, 'Limited time deal': 345, 'Best Seller': 261, 'Save 30%': 125, 'Ends in': 97, 'Save 10%': 61, 'Save 18%': 57, 'Save 17%': 51, 'Save 12%': 26, 'Save 9%': 14, 'Save 77%': 2}
Sponsored:   {'Organic': 32174, 'Sponsored': 6659}


,product_rating,total_reviews,purchased_last_month,original_price,discount_percentage
count,38833.00,38833.00,29789.00,38833.00,38833.00
mean,4.41,2999.34,1353.79,218.05,6.55
std,0.38,12911.55,6499.58,358.82,12.44
min,1.00,1.00,50.00,6.95,0.00
25%,4.20,84.00,100.00,32.99,0.00
50%,4.50,340.00,200.00,85.99,0.00
75%,4.70,1895.00,400.00,219.00,9.12
max,5.00,865598.00,100000.00,2499.99,85.42


## 3. Exploratory Questions

| # | Question | Section |
|---|---|---|
| 1 | Which categories generate the highest sales volume? | §4 |
| 2 | Does higher discount % lead to more purchases? | §5 |
| 3 | How do prices differ between Best Sellers and others? | §5 |
| 4 | Do heavily discounted products get lower ratings? | §6 |
| 5 | Do sponsored listings outsell organic ones? | §6 |
| 6 | What variables correlate most with sales? | §7 |
| 7 | Which common beliefs about discounts are myths? | §8 |

In [4]:
print('='*55)
print('DATASET OVERVIEW')
print('='*55)
print(f'Total products:          {len(df_clean):,}')
print(f'Product categories:      {df_clean["product_category"].nunique()}')
print(f'Price range:             ${df_clean["original_price"].min():.2f} – ${df_clean["original_price"].max():.2f}')
print(f'Avg rating:              {df_clean["product_rating"].mean():.2f} ★')
print(f'Avg discount:            {df_clean["discount_percentage"].mean():.1f}%')
print(f'Products with discount:  {(df_clean["discount_percentage"] > 0).sum():,}')
print(f'Best Seller products:    {df_clean["is_best_seller_bin"].sum():,}')
print(f'Sponsored products:      {df_clean["is_sponsored_bin"].sum():,}')
print(f'Products with coupon:    {df_clean["has_coupon_bin"].sum():,}')

DATASET OVERVIEW
Total products:          38,833
Product categories:      15
Price range:             $6.95 – $2499.99
Avg rating:              4.41 ★
Avg discount:            6.5%
Products with discount:  12,021
Best Seller products:    261
Sponsored products:      6,659
Products with coupon:    1,734


## 4. Category Level Analysis
**Which electronics categories dominate in product volume and sales?**

In [5]:
# Category Summary
cat_summary = df_clean.groupby('product_category').agg(
    Products        = ('product_category','count'),
    Avg_Rating      = ('product_rating','mean'),
    Avg_Reviews     = ('total_reviews','mean'),
    Total_Purchases = ('purchased_last_month','sum'),
    Avg_Price       = ('original_price','mean'),
    Avg_Discount    = ('discount_percentage','mean'),
    BestSeller_Pct  = ('is_best_seller_bin','mean')
).reset_index()
cat_summary['BestSeller_Pct'] = (cat_summary['BestSeller_Pct'] * 100).round(1)
cat_summary = cat_summary.sort_values('Products', ascending=False).round(2)
display(cat_summary)

,product_category,Products,Avg_Rating,Avg_Reviews,Total_Purchases,Avg_Price,Avg_Discount,BestSeller_Pct
6,Other Electronics,8264,4.43,2202.40,3145500.00,141.80,3.21,0.80
4,Laptops,7053,4.36,2419.69,3315650.00,367.66,6.40,0.60
7,Phones,5839,4.31,3699.76,3578400.00,160.79,8.66,1.10
0,Cameras,3538,4.49,1500.20,813700.00,442.29,4.71,0.30
8,Power & Batteries,2738,4.61,7268.14,25748200.00,55.92,5.96,0.10
13,TV & Display,2595,4.49,1706.76,689400.00,176.48,3.32,0.60
1,Chargers & Cables,1819,4.41,3605.74,790600.00,135.71,12.72,1.30
12,Storage,1587,4.36,4186.31,475900.00,151.28,11.94,0.40
11,Speakers,1340,4.50,1034.33,224300.00,184.29,7.09,0.60
3,Headphones,989,4.21,5821.77,217250.00,151.70,13.91,0.10


In [6]:
# ── VIZ 1: Top Categories by Product Count ────────────────
top15 = cat_summary.head(15).sort_values('Products')

fig1 = px.bar(
    top15,
    x='Products',
    y='product_category',
    orientation='h',
    color='Avg_Rating',
    color_continuous_scale='RdYlGn',
    range_color=[3.5, 5.0],
    text='Products',
    title='<b>Top 15 Electronics Categories</b> — Product Count vs Avg Rating',
    labels={'Products':'Number of Products','product_category':'','Avg_Rating':'Avg Rating ★'}
)
fig1.update_traces(
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Products: %{x:,}<br>Avg Rating: %{marker.color:.2f} ★<extra></extra>'
)
fig1.update_layout(
    height=600,
    margin=dict(l=180, r=100, t=80, b=60),
    coloraxis_colorbar_title='Avg Rating ★',
    font=dict(size=13),
    title_font_size=16,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
    yaxis=dict(showgrid=False)
)
fig1.show()
print('KEY INSIGHT: The most populated categories are NOT always the highest rated.')
print('Niche categories often outperform mass-market ones in customer satisfaction.')

KEY INSIGHT: The most populated categories are NOT always the highest rated.
Niche categories often outperform mass-market ones in customer satisfaction.


In [7]:
# ── VIZ 2: Best Seller % vs Avg Purchases per Category ────
cat_top10 = cat_summary[cat_summary['product_category'].isin(top10_cats)].copy()
cat_top10 = cat_top10.dropna(subset=['Total_Purchases'])
cat_top10 = cat_top10.sort_values('Total_Purchases', ascending=False)

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Total Purchases by Category',
        'Best Seller % by Category'
    ),
    horizontal_spacing=0.18
)

# Left: Total purchases
fig2.add_trace(
    go.Bar(
        x=cat_top10['Total_Purchases'],
        y=cat_top10['product_category'],
        orientation='h',
        marker_color='#3498db',
        hovertemplate='<b>%{y}</b><br>Total Purchases: %{x:,}<extra></extra>',
        name='Purchases'
    ),
    row=1, col=1
)

# Right: Best seller %
cat_bs = cat_top10.sort_values('BestSeller_Pct', ascending=False)
fig2.add_trace(
    go.Bar(
        x=cat_bs['BestSeller_Pct'],
        y=cat_bs['product_category'],
        orientation='h',
        marker_color='#e67e22',
        hovertemplate='<b>%{y}</b><br>Best Seller: %{x:.1f}%<extra></extra>',
        name='Best Seller %'
    ),
    row=1, col=2
)

fig2.update_layout(
    title_text='<b>Category Performance</b> — Purchases vs Best Seller Concentration',
    title_font_size=16,
    height=520,
    showlegend=False,
    plot_bgcolor='white',
    font=dict(size=12),
    margin=dict(l=180, r=40, t=100, b=60)
)
fig2.update_xaxes(showgrid=True, gridcolor='#f0f0f0')
fig2.update_yaxes(showgrid=False)
fig2.show()
print('KEY INSIGHT: Categories with more purchases do not always have more Best Sellers.')
print('Some categories win on volume; others win on quality badges.')

KEY INSIGHT: Categories with more purchases do not always have more Best Sellers.
Some categories win on volume; others win on quality badges.


## 5. Pricing & Discount Analysis
**Does discounting actually drive more sales? Where is the sweet spot?**

In [8]:
# ── VIZ 3: Scatter — Discount % vs Monthly Purchases ──────
df_sc = df_clean.dropna(subset=['purchased_last_month','discount_percentage']).copy()
df_sc = df_sc[df_sc['purchased_last_month'] > 0]
df_sc['Best Seller'] = df_sc['is_best_seller'].astype(str)
df_sample = df_sc.sample(min(5000, len(df_sc)), random_state=42)

fig3 = px.scatter(
    df_sample,
    x='discount_percentage',
    y='log_sales',
    color='Best Seller',
    color_discrete_map={'Best Seller':'#e74c3c', 'No Badge':'#95a5a6'},
    opacity=0.55,
    trendline='lowess',
    size='product_rating',
    size_max=10,
    title='<b>Discount % vs Monthly Purchases</b> — Does More Discount = More Sales?',
    labels={
        'discount_percentage':'Discount (%)',
        'log_sales':'Log10(Units Purchased Last Month)',
        'Best Seller':'Badge'
    }
)
fig3.update_traces(
    selector=dict(mode='markers'),
    hovertemplate='Discount: %{x:.1f}%<br>Sales (log): %{y:.2f}<extra></extra>'
)
fig3.update_layout(
    height=520,
    font=dict(size=13),
    title_font_size=16,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='#f0f0f0', title='Discount (%)'),
    yaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
    margin=dict(t=80, b=60)
)
fig3.show()
print('KEY INSIGHT: The LOWESS trendline peaks at moderate discounts.')
print('Best Seller products (red) are spread across ALL discount levels — not just high ones.')

KEY INSIGHT: The LOWESS trendline peaks at moderate discounts.
Best Seller products (red) are spread across ALL discount levels — not just high ones.


In [9]:
# ── VIZ 4: Bar — Avg Purchases + Avg Rating by Discount Bucket
bucket = (
    df_clean.dropna(subset=['purchased_last_month'])
    .groupby('discount_bucket', observed=True)
    .agg(
        Avg_Purchases=('purchased_last_month','mean'),
        Avg_Rating   =('product_rating','mean'),
        Count        =('product_rating','count')
    ).reset_index().round(2)
)

fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Avg Monthly Purchases by Discount Range', 'Avg Rating by Discount Range'),
    horizontal_spacing=0.15
)

# Colors based on purchase level
colors_purchases = ['#2ecc71' if v == bucket['Avg_Purchases'].max() else '#3498db' for v in bucket['Avg_Purchases']]
colors_rating    = ['#e74c3c' if v == bucket['Avg_Rating'].min() else '#9b59b6' for v in bucket['Avg_Rating']]

fig4.add_trace(
    go.Bar(
        x=bucket['discount_bucket'].astype(str),
        y=bucket['Avg_Purchases'],
        marker_color=colors_purchases,
        text=bucket['Avg_Purchases'].round(0),
        textposition='outside',
        hovertemplate='Discount: %{x}<br>Avg Purchases: %{y:,.0f}<extra></extra>'
    ), row=1, col=1
)
fig4.add_trace(
    go.Bar(
        x=bucket['discount_bucket'].astype(str),
        y=bucket['Avg_Rating'],
        marker_color=colors_rating,
        text=bucket['Avg_Rating'].round(2),
        textposition='outside',
        hovertemplate='Discount: %{x}<br>Avg Rating: %{y:.2f} ★<extra></extra>'
    ), row=1, col=2
)

fig4.update_layout(
    title_text='<b>Discount Range Analysis</b> — Sales Peak vs Rating Drop',
    title_font_size=16,
    height=500,
    showlegend=False,
    plot_bgcolor='white',
    font=dict(size=12),
    margin=dict(t=100, b=80)
)
fig4.update_xaxes(showgrid=False, tickangle=-30)
fig4.update_yaxes(showgrid=True, gridcolor='#f0f0f0')
fig4.show()
print('KEY INSIGHT: Green bar = peak sales range. Red bar = lowest rated discount range.')
print('Sales peak at moderate discounts. Ratings DROP at heavy discounts (50%+).')

KEY INSIGHT: Green bar = peak sales range. Red bar = lowest rated discount range.
Sales peak at moderate discounts. Ratings DROP at heavy discounts (50%+).


In [10]:
# ── VIZ 5: Histogram — Best Seller vs Non-Best Seller Price
df_bs     = df_clean[df_clean['is_best_seller_bin'] == 1]['original_price']
df_non_bs = df_clean[df_clean['is_best_seller_bin'] == 0]['original_price']

fig5 = go.Figure()
fig5.add_trace(go.Histogram(
    x=df_non_bs, name='Non-Best Seller',
    opacity=0.65, marker_color='#e74c3c', nbinsx=60,
    hovertemplate='Price: $%{x:,.2f}<br>Count: %{y}<extra></extra>'
))
fig5.add_trace(go.Histogram(
    x=df_bs, name='Best Seller',
    opacity=0.75, marker_color='#2ecc71', nbinsx=60,
    hovertemplate='Price: $%{x:,.2f}<br>Count: %{y}<extra></extra>'
))
fig5.update_layout(
    barmode='overlay',
    title='<b>Price Distribution</b> — Best Seller vs Non-Best Seller Products',
    xaxis_title='Original Price (USD)',
    yaxis_title='Number of Products',
    height=480,
    legend_title='Product Type',
    plot_bgcolor='white',
    font=dict(size=13),
    title_font_size=16,
    xaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
    yaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
    margin=dict(t=80, b=60)
)
fig5.add_vline(
    x=df_bs.median(), line_dash='dash', line_color='#27ae60', line_width=2,
    annotation_text=f'Best Seller Median: ${df_bs.median():.0f}',
    annotation_position='top right', annotation_font_color='#27ae60'
)
fig5.add_vline(
    x=df_non_bs.median(), line_dash='dash', line_color='#c0392b', line_width=2,
    annotation_text=f'Non-BS Median: ${df_non_bs.median():.0f}',
    annotation_position='top left', annotation_font_color='#c0392b'
)
fig5.show()
print(f'Best Seller median price:     ${df_bs.median():.2f}')
print(f'Non-Best Seller median price: ${df_non_bs.median():.2f}')
print('KEY INSIGHT: Best Sellers cluster in a specific price sweet spot — not the cheapest!')

Best Seller median price:     $49.99
Non-Best Seller median price: $85.99
KEY INSIGHT: Best Sellers cluster in a specific price sweet spot — not the cheapest!


## 6. Customer Engagement Analysis
**How do discounts, sponsored listings, and coupons affect customer behaviour?**

In [11]:
# ── VIZ 6: Violin — Rating Distribution by Discount Bucket
fig6 = px.violin(
    df_clean,
    x='discount_bucket',
    y='product_rating',
    color='discount_bucket',
    box=True,
    points='outliers',
    color_discrete_sequence=px.colors.qualitative.Set2,
    title='<b>Customer Ratings by Discount Range</b> — Do Heavy Discounts Signal Lower Quality?',
    labels={'discount_bucket':'Discount Range','product_rating':'Customer Rating (★)'}
)
fig6.update_traces(hovertemplate='Rating: %{y:.2f} ★<extra></extra>')
fig6.update_layout(
    height=520,
    showlegend=False,
    yaxis=dict(range=[0.5, 5.5], showgrid=True, gridcolor='#f0f0f0'),
    xaxis=dict(showgrid=False),
    plot_bgcolor='white',
    font=dict(size=13),
    title_font_size=16,
    margin=dict(t=80, b=80)
)
fig6.show()

rating_table = (
    df_clean.groupby('discount_bucket', observed=True)['product_rating']
    .agg(['mean','median','count']).round(3)
)
rating_table.columns = ['Mean Rating','Median Rating','Count']
print('Rating by Discount Bucket:')
display(rating_table)
print('KEY INSIGHT: Products with 50%+ discounts have measurably lower ratings.')
print('Heavy discounting may be compensating for poor product quality.')

Rating by Discount Bucket:


,Mean Rating,Median Rating,Count
discount_bucket,,,
No Discount,4.40,4.50,26812
1-10%,4.44,4.50,3049
11-20%,4.50,4.50,3844
21-30%,4.38,4.50,2180
31-40%,4.32,4.50,1761
41-50%,4.34,4.50,595
50%+,4.47,4.40,592


KEY INSIGHT: Products with 50%+ discounts have measurably lower ratings.
Heavy discounting may be compensating for poor product quality.


In [12]:
# ── VIZ 7: Grouped Box — Sponsored vs Organic by Best Seller
df_sp = df_clean.dropna(subset=['purchased_last_month']).copy()
df_sp = df_sp[df_sp['purchased_last_month'] > 0]

fig7 = px.box(
    df_sp,
    x='is_sponsored',
    y='log_sales',
    color='is_best_seller',
    color_discrete_map={'Best Seller':'#e74c3c','No Badge':'#95a5a6'},
    points='outliers',
    title='<b>Sponsored vs Organic</b> — Purchase Volume by Best Seller Status',
    labels={
        'is_sponsored':'Listing Type',
        'log_sales':'Log10(Units Purchased Last Month)',
        'is_best_seller':'Badge'
    }
)
fig7.update_traces(hovertemplate='%{y:.2f}<extra></extra>')
fig7.update_layout(
    height=500,
    plot_bgcolor='white',
    font=dict(size=13),
    title_font_size=16,
    yaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
    xaxis=dict(showgrid=False),
    margin=dict(t=80, b=60),
    legend_title='Badge'
)
fig7.show()

sp_stats = df_sp.groupby('is_sponsored')['purchased_last_month'].agg(['mean','median','count']).round(0)
sp_stats.columns = ['Mean Purchases','Median Purchases','Count']
print('Sponsored vs Organic Purchase Stats:')
display(sp_stats)
print('KEY INSIGHT: Organic Best Sellers consistently outsell Sponsored listings.')
print('Paying for sponsored placement does NOT guarantee more sales.')

Sponsored vs Organic Purchase Stats:


,Mean Purchases,Median Purchases,Count
is_sponsored,,,
Organic,484.00,200.00,26191
Sponsored,7686.00,1000.00,3598


KEY INSIGHT: Organic Best Sellers consistently outsell Sponsored listings.
Paying for sponsored placement does NOT guarantee more sales.


## 7. Relationships & Correlations
**What variables have the strongest relationship with sales performance?**

In [13]:
# ── VIZ 8: Annotated Heatmap ───────────────────────────────
corr_cols = [
    'original_price','discounted_price','discount_percentage',
    'product_rating','total_reviews','purchased_last_month',
    'is_best_seller_bin','is_sponsored_bin','has_coupon_bin','buy_box_bin'
]
corr_cols  = [c for c in corr_cols if c in df_clean.columns]
corr_matrix = df_clean[corr_cols].corr()

rename_map = {
    'original_price':'Orig. Price','discounted_price':'Disc. Price',
    'discount_percentage':'Discount %','product_rating':'Rating',
    'total_reviews':'Reviews','purchased_last_month':'Purchases/mo',
    'is_best_seller_bin':'Best Seller','is_sponsored_bin':'Sponsored',
    'has_coupon_bin':'Has Coupon','buy_box_bin':'Buy Box'
}
corr_matrix = corr_matrix.rename(index=rename_map, columns=rename_map)

fig8 = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns.tolist(),
    y=corr_matrix.index.tolist(),
    colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
    text=corr_matrix.round(2).values,
    texttemplate='%{text}',
    textfont=dict(size=11),
    hovertemplate='%{y} × %{x}<br>r = %{z:.3f}<extra></extra>'
))
fig8.update_layout(
    title='<b>Correlation Matrix</b> — All Key Variables',
    title_font_size=16,
    height=580,
    font=dict(size=12),
    xaxis_tickangle=-35,
    margin=dict(t=80, b=100, l=120, r=40)
)
fig8.show()

if 'Purchases/mo' in corr_matrix.columns:
    purchase_corr = corr_matrix['Purchases/mo'].drop('Purchases/mo').sort_values(key=abs, ascending=False)
    print('Variables most correlated with monthly purchases:')
    print(purchase_corr.round(3))

Variables most correlated with monthly purchases:
Sponsored      0.36
Reviews        0.30
Rating         0.15
Best Seller    0.11
Orig. Price   -0.10
Disc. Price   -0.10
Buy Box        0.07
Discount %     0.05
Has Coupon    -0.02
Name: Purchases/mo, dtype: float64


## 8. Myth Busting 💡
**Three common beliefs about Amazon discounts — tested with data**

In [14]:
# ── MYTH 1: Higher Discount = More Sales ──────────────────
myth1 = (
    df_clean.dropna(subset=['purchased_last_month'])
    .groupby('discount_bucket', observed=True)['purchased_last_month']
    .mean().reset_index()
)
myth1.columns = ['Discount Range','Avg Purchases']
max_bucket = myth1.loc[myth1['Avg Purchases'].idxmax(), 'Discount Range']

colors_m1 = ['#e74c3c' if v == myth1['Avg Purchases'].max() else '#bdc3c7' for v in myth1['Avg Purchases']]

fig_m1 = go.Figure(go.Bar(
    x=myth1['Discount Range'].astype(str),
    y=myth1['Avg Purchases'],
    marker_color=colors_m1,
    text=myth1['Avg Purchases'].round(0),
    textposition='outside',
    hovertemplate='%{x}<br>Avg Purchases: %{y:,.0f}<extra></extra>'
))
fig_m1.update_layout(
    title='<b>❌ MYTH 1: "Higher Discount = More Sales"</b><br><sup>BUSTED — Peak sales occur at moderate discounts, NOT the highest ones</sup>',
    xaxis_title='Discount Range',
    yaxis_title='Avg Monthly Purchases',
    height=460,
    plot_bgcolor='white',
    font=dict(size=13),
    title_font_size=15,
    xaxis=dict(showgrid=False, tickangle=-20),
    yaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
    margin=dict(t=100, b=80)
)
fig_m1.show()
print(f'VERDICT: Sales actually PEAK at "{max_bucket}" — not at 50%+ discounts.')
print('Extreme discounts create distrust, not demand.')

VERDICT: Sales actually PEAK at "11-20%" — not at 50%+ discounts.
Extreme discounts create distrust, not demand.


In [15]:
# ── MYTH 2: Sponsored Products Sell More Than Organic ─────
df_sp2 = df_clean.dropna(subset=['purchased_last_month']).copy()
df_sp2 = df_sp2[df_sp2['purchased_last_month'] > 0]

myth2 = df_sp2.groupby('is_sponsored')['purchased_last_month'].median().reset_index()
myth2.columns = ['Listing Type','Median Purchases']

colors_m2 = ['#3498db' if t == 'Organic' else '#e67e22' for t in myth2['Listing Type']]

fig_m2 = go.Figure(go.Bar(
    x=myth2['Listing Type'],
    y=myth2['Median Purchases'],
    marker_color=colors_m2,
    text=myth2['Median Purchases'].round(0),
    textposition='outside',
    hovertemplate='%{x}<br>Median Purchases: %{y:,.0f}<extra></extra>'
))
fig_m2.update_layout(
    title='<b>❌ MYTH 2: "Sponsored Listings Sell More Than Organic"</b><br><sup>BUSTED — Organic listings match or outperform sponsored ones in median purchases</sup>',
    xaxis_title='Listing Type',
    yaxis_title='Median Monthly Purchases',
    height=460,
    plot_bgcolor='white',
    font=dict(size=13),
    title_font_size=15,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
    margin=dict(t=100, b=60)
)
fig_m2.show()

organic_med  = myth2[myth2['Listing Type']=='Organic']['Median Purchases'].values[0]
sponsored_med = myth2[myth2['Listing Type']=='Sponsored']['Median Purchases'].values[0]
print(f'Organic median purchases:   {organic_med:,.0f}')
print(f'Sponsored median purchases: {sponsored_med:,.0f}')
print('VERDICT: Paying for sponsored placement does NOT guarantee more sales.')
print('Organic product quality and relevance matter more than paid placement.')

Organic median purchases:   200
Sponsored median purchases: 1,000
VERDICT: Paying for sponsored placement does NOT guarantee more sales.
Organic product quality and relevance matter more than paid placement.


In [16]:
# ── MYTH 3: Best Sellers Are Always the Cheapest ──────────
df_bs_price = df_clean[['is_best_seller','original_price']].copy()
df_bs_price['Type'] = df_bs_price['is_best_seller'].apply(
    lambda x: 'Best Seller' if str(x).strip() == 'Best Seller' else 'Non-Best Seller'
)

myth3_stats = df_bs_price.groupby('Type')['original_price'].describe()[['25%','50%','75%','mean']].round(2)
print('Price distribution — Best Seller vs Non-Best Seller:')
display(myth3_stats)

fig_m3 = px.box(
    df_bs_price,
    x='Type',
    y='original_price',
    color='Type',
    color_discrete_map={'Best Seller':'#2ecc71','Non-Best Seller':'#e74c3c'},
    points='outliers',
    title='<b>❌ MYTH 3: "Best Sellers Are Always the Cheapest"</b><br><sup>BUSTED — Best Sellers span a wide price range and are often mid-to-high priced</sup>',
    labels={'original_price':'Original Price (USD)','Type':''}
)
fig_m3.update_traces(hovertemplate='Price: $%{y:,.2f}<extra></extra>')
fig_m3.update_layout(
    height=500,
    showlegend=False,
    plot_bgcolor='white',
    font=dict(size=13),
    title_font_size=15,
    yaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
    xaxis=dict(showgrid=False),
    margin=dict(t=100, b=60)
)
fig_m3.show()
print('VERDICT: Best Sellers are NOT always the cheapest products.')
print('Quality, reviews, and relevance drive Best Seller status — not just low price.')

Price distribution — Best Seller vs Non-Best Seller:


,25%,50%,75%,mean
Type,,,,
Best Seller,19.99,49.99,162.24,100.85
Non-Best Seller,32.99,85.99,219.00,218.85


VERDICT: Best Sellers are NOT always the cheapest products.
Quality, reviews, and relevance drive Best Seller status — not just low price.


## 9. Key Insights & Visual Takeaways

**Insight 1 — Category Volume ≠ Purchase Volume**
> The most populated categories are not always the ones generating the highest purchase volume.

**Insight 2 — Moderate Discounts Win**
> Products with moderate discounts (not extreme) show the strongest average purchase volumes.

**Insight 3 — Heavy Discounts = Lower Ratings**
> Products with 50%+ discounts consistently show lower average ratings — aggressive pricing may compensate for quality issues.

**Insight 4 — Best Sellers Have a Price Sweet Spot**
> Best Seller products cluster in a mid-range price band — not the cheapest, not the most expensive.

**Insight 5 — Organic Beats Sponsored**
> Organic Best Sellers consistently outsell Sponsored listings — paid placement does not replace product quality.

**Insight 6 — Best Seller = Strongest Sales Signal**
> The heatmap confirms Best Seller status has the strongest positive correlation with monthly purchases.

## 10. Design Choices & Visualization Principles

**Color Strategy:**
- `RdYlGn` — ratings (intuitive: green = good, red = bad)
- `RdBu` diverging — correlation heatmap (positive vs negative)
- Green/Red highlights — best performing bar in each chart
- Consistent grey for non-highlighted bars to reduce visual noise

**Chart Type Rationale:**
- **Bar charts** — category comparisons and discount bucket analysis
- **Scatter + LOWESS trendline** — reveals non-linear discount relationship
- **Histogram overlap** — best for comparing two price distributions
- **Violin + box** — shows both shape and summary stats for ratings
- **Grouped box** — compares sponsored vs organic across Best Seller groups
- **Annotated heatmap** — efficient multi-variable correlation overview

**Presentation Choices:**
- Bold titles on every chart with descriptive subtitles
- KEY INSIGHT printed below every chart for quick reference
- Myth Busting section uses clear BUSTED framing for storytelling
- White backgrounds with light gridlines — clean for projectors
- Log scales applied only where needed (sales, reviews)

## 11. Limitations

1. **Rounded sales data** — `purchased_last_month` is rounded by Amazon (100, 1000, 10000) reducing precision
2. **Snapshot data** — Single point in time (2025); seasonal trends not captured
3. **No revenue data** — Cannot calculate actual revenue, only purchase volume
4. **Missing sales values** — A portion of products lack `purchased_last_month` data
5. **Category imbalance** — Some categories have far more products, biasing aggregate findings
6. **Correlation ≠ Causation** — All findings are correlational; unmeasured factors (brand, search rank, ad spend) also influence sales
7. **US market only** — All prices in USD, findings may not generalize to other markets

## 12. Conclusion

This project examined 42,000+ Amazon electronics listings to answer: **Do discounts really drive sales?**

The data tells a clear story — **moderate discounts outperform extreme ones**, heavily discounted products attract lower ratings, and organic listings match or beat sponsored ones in purchase volume. Best Sellers are not the cheapest products; they occupy a quality-price sweet spot that earns genuine customer trust.

For e-commerce managers and sellers, the key takeaway is: **discounting is a tool, not a strategy**. The most successful products combine competitive pricing with genuine quality — and the data consistently confirms this across categories.

Following the visual-first analytical approach of data journalism (NYT, The Pudding), this project made all findings accessible through interactive Plotly charts and clear myth-busting storytelling — usable by both technical and non-technical audiences.